<a href="https://www.kaggle.com/code/michaelenoh/multi-class-prediction-the-risk-of-obesity?scriptVersionId=345879732" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Multi-Class Prediction of Obesity Risk II
### Kaggle: Multi-Class Prediction of Obesity Risk (Playground Series S4E2)
Decision tree, bagged, random forest, and boosted classification models, plus a regularized logistic regression baseline.

**Requires** `train.csv`, `test.csv`, and `sample_submission.csv` (from the Kaggle competition) to be in the same folder as this notebook.

## Introduction

Obesity-risk classification constitutes a multiclass prediction problem that incorporates demographic characteristics, physical measurements, eating habits, and lifestyle behaviors. The dataset analyzed in this study was designed to estimate obesity levels based on eating habits and physical condition among individuals from Colombia, Peru, and Mexico (Palechor & de la Hoz Manotas, 2019). Because the relationships between these predictors and obesity classifications may be nonlinear and interactive, tree-based methods are suitable alternatives to conventional linear classification models.

This study compares decision tree, bagging, random forest, and hyperparameter-tuned XGBoost models with an L2-regularized multinomial logistic regression baseline. Random forests extend bagging by constructing decorrelated trees from bootstrap samples and randomly selected predictor subsets, thereby reducing prediction variance (Breiman, 2001). XGBoost implements scalable gradient-boosted trees with shrinkage, subsampling, and regularization to control model complexity and enhance predictive performance (Chen & Guestrin, 2016). We evaluate the models using five-fold stratified cross-validation and class-specific performance measures, and generate four Kaggle-compatible submission files for leaderboard evaluation.

## Kaggle: Multi-class Prediction of Obesity Risk

Competition: *Playground Series S4E2 — Multi-Class Prediction of Obesity Risk*
(https://www.kaggle.com/competitions/playground-series-s4e2/overview)

Target: `NObeyesdad` (7 obesity-risk classes). We build a decision tree, a
bagged tree ensemble, a random forest, and a boosted model, and also fit a
regularized (L2-penalized) multinomial logistic regression as the "simple
method" comparison requested by ISLR Q12.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## Load Dataset

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_DIR = Path("/kaggle/input")
csv_files = list(INPUT_DIR.rglob("*.csv"))

def locate_csv(filename):
    matches = [
        path for path in csv_files
        if path.name.lower() == filename.lower()
    ]

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found under {INPUT_DIR}. "
            "Add the competition dataset through Add Input."
        )

    print(f"{filename}: {matches[0]}")
    return matches[0]

train_path = locate_csv("train.csv")
test_path = locate_csv("test.csv")
sample_path = locate_csv("sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        "The /kaggle/input directory is unavailable. "
        "Make sure this notebook is running on Kaggle."
    )

csv_files = sorted(INPUT_DIR.rglob("*.csv"))

print("CSV files available:")
for file in csv_files:
    print(file)

In [ ]:
print(train.shape, test.shape)
train.head()

In [ ]:
target = "NObeyesdad"
id_col = "id"

cat_features = train.select_dtypes(include="object").columns.tolist()
cat_features.remove(target)
num_features = [c for c in train.columns if c not in cat_features + [target, id_col]]

print("Categorical:", cat_features)
print("Numeric:", num_features)
train[target].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb

X_full = train.drop(columns=[target, id_col])
y_full_raw = train[target]
X_kaggle_test = test.drop(columns=[id_col])

le_target = LabelEncoder()
y_full = le_target.fit_transform(y_full_raw)

cat_pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
], remainder="passthrough")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Model 1: Decision Tree

In [ ]:
tree_pipe = Pipeline([
    ("pre", cat_pre),
    ("model", DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=42)),
])
tree_scores = cross_val_score(tree_pipe, X_full, y_full, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"Decision Tree CV accuracy: {tree_scores.mean():.4f} (+/- {tree_scores.std():.4f})")

## Model 2: Bagging

In [ ]:
bag_pipe = Pipeline([
    ("pre", cat_pre),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=300, random_state=42, n_jobs=-1
    )),
])
bag_scores = cross_val_score(bag_pipe, X_full, y_full, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"Bagging CV accuracy: {bag_scores.mean():.4f} (+/- {bag_scores.std():.4f})")

## Model 3: Random Forest

In [ ]:
rf_pipe2 = Pipeline([
    ("pre", cat_pre),
    ("model", RandomForestClassifier(
        n_estimators=500, max_features="sqrt", random_state=42, n_jobs=-1
    )),
])
rf_scores = cross_val_score(rf_pipe2, X_full, y_full, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"Random Forest CV accuracy: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")

## Model 4: Boosting (XGBoost)
RandomizedSearchCV evaluates multiple hyperparameter combinations using stratified cross-validation. The search tunes model complexity, learning rate, sampling, and regularization. Its best fitted pipeline is retained as xgb_pipe and used in all subsequent evaluation and Kaggle-submission steps.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

xgb_base_pipe = Pipeline([
    ("pre", cat_pre),
    ("model", xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(le_target.classes_),
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=1,
    )),
])

# A compact search space keeps the Kaggle runtime manageable while tuning the
# principal complexity, shrinkage, sampling, and regularization parameters.
xgb_param_distributions = {
    "model__n_estimators": [250, 350, 450, 600],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.03, 0.05, 0.075, 0.10],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.75, 0.85, 1.00],
    "model__colsample_bytree": [0.75, 0.90, 1.00],
    "model__reg_alpha": [0.0, 0.01, 0.10],
    "model__reg_lambda": [1.0, 2.0, 5.0],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base_pipe,
    param_distributions=xgb_param_distributions,
    n_iter=15,
    scoring="accuracy",
    cv=cv,
    refit=True,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=False,
)

xgb_search.fit(X_full, y_full)
xgb_pipe = xgb_search.best_estimator_

print("Best XGBoost parameters:")
print(xgb_search.best_params_)
print(f"Best tuned 5-fold CV accuracy: {xgb_search.best_score_:.4f}")

# Use the search score in the model-comparison table. Re-running
# cross_val_score on the selected parameters would reuse the same observations
# employed during tuning and would not constitute a fully nested evaluation.
xgb_scores = xgb_search.cv_results_["mean_test_score"]
xgb_best_std = xgb_search.cv_results_["std_test_score"][xgb_search.best_index_]


## Simple-method comparison: Regularized (L2) Multinomial Logistic Regression

In [ ]:
logreg_pipe = Pipeline([
    ("pre", cat_pre),
    ("scale", StandardScaler(with_mean=False)),
    ("model", LogisticRegression(
        penalty="l2", C=1.0, max_iter=2000, solver="lbfgs", random_state=42
    )),
])
logreg_scores = cross_val_score(logreg_pipe, X_full, y_full, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"Regularized Logistic Regression CV accuracy: {logreg_scores.mean():.4f} (+/- {logreg_scores.std():.4f})")

## Model Comparison Summary

In [ ]:
q3_results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Bagging",
        "Random Forest",
        "Boosting (Tuned XGBoost)",
        "Regularized Logistic Regression"
    ],
    "CV Accuracy": [
        tree_scores.mean(),
        bag_scores.mean(),
        rf_scores.mean(),
        xgb_search.best_score_,
        logreg_scores.mean()
    ],
    "CV Std": [
        tree_scores.std(),
        bag_scores.std(),
        rf_scores.std(),
        xgb_best_std,
        logreg_scores.std()
    ],
}).sort_values(
    "CV Accuracy",
    ascending=False
).reset_index(drop=True)

q3_results

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.barh(q3_results["Model"], q3_results["CV Accuracy"], color="#C44E52")
ax.set_xlabel("5-Fold CV Accuracy")
ax.set_title("Question 3: Model Comparison (Obesity Risk)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("q3_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Confusion Matrix and Interpretation (Best Model: Tuned XGBoost)

In [ ]:
cv_preds = cross_val_predict(xgb_pipe, X_full, y_full, cv=cv, n_jobs=-1)
cm = confusion_matrix(y_full, cv_preds)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(le_target.classes_)))
ax.set_yticks(range(len(le_target.classes_)))
ax.set_xticklabels(le_target.classes_, rotation=45, ha="right")
ax.set_yticklabels(le_target.classes_)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix: Boosting (Out-of-Fold CV Predictions)")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.tight_layout()
plt.savefig("q3_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(classification_report(y_full, cv_preds, target_names=le_target.classes_))

## Feature Importance (Boosting model)

In [ ]:
xgb_pipe.fit(X_full, y_full)
feat_names = xgb_pipe.named_steps["pre"].get_feature_names_out()
importances_xgb = pd.Series(
    xgb_pipe.named_steps["model"].feature_importances_, index=feat_names
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(7.5, 5))
importances_xgb.plot(kind="barh", ax=ax, color="#8172B2")
ax.invert_yaxis()
ax.set_title("Top 15 Feature Importances — Boosting (XGBoost)")
plt.tight_layout()
plt.savefig("q3_xgb_importance.png", dpi=150, bbox_inches="tight")
plt.show()
importances_xgb

The XGBoost importance measure assigned substantial importance to gender indicators and weight, followed by vegetable-consumption frequency and eating-related categorical levels. These importance values describe predictive contributions within this fitted model; they do not establish causal effects.

## Final Models Submissions

In [ ]:
final_models = {
    "decision_tree": tree_pipe,
    "bagging": bag_pipe,
    "random_forest": rf_pipe2,
    "boosting": xgb_pipe,
}

for name, pipe in final_models.items():
    pipe.fit(X_full, y_full)
    preds = pipe.predict(X_kaggle_test)
    preds_labels = le_target.inverse_transform(preds)
    sub = pd.DataFrame({id_col: test[id_col], target: preds_labels})
    fname = f"submission_{name}.csv"
    sub.to_csv(fname, index=False)
    print(f"Saved {fname}  |  shape={sub.shape}")

In [ ]:
print("\nFinal 5-fold CV accuracy summary:")
print(q3_results.to_string(index=False))

Logistic regression assumes that predictors have linear relationships with the multinomial log-odds and may be affected by severe multicollinearity. Tree-based models do not require normally distributed predictors or linear relationships, but their results depend on representative observations and appropriate control of model complexity. Stratified cross-validation, shrinkage, subsampling, and regularization were used to reduce the risk of overfitting. Feature importance was interpreted as predictive rather than causal.

## Interpretation and Conclusions

The hyperparameter-tuned XGBoost model achieved the highest five-fold cross-validation accuracy (0.9091), followed by random forest (0.8929), bagging (0.8918), decision tree (0.8715), and L2-regularized logistic regression (0.8625). These results indicate that ensemble tree models outperform both the single decision tree and the logistic regression baseline. The improvement observed with random forest aligns with Breiman’s (2001) assertion that aggregating decorrelated trees reduces prediction variance.

RandomizedSearchCV evaluated 15 XGBoost parameter combinations using five-fold stratified cross-validation. The optimal configuration included 350 trees, a maximum depth of 5, a learning rate of 0.05, subsampling of 0.85, column subsampling of 0.90, a minimum child weight of 3, and an L2 regularization value of 5.0. These parameters illustrate the shrinkage, sampling, and regularization strategies XGBoost uses to control model complexity and improve generalization (Chen & Guestrin, 2016).

The tuned XGBoost model classified Obesity Type III with near-perfect accuracy but exhibited lower precision and recall for Overweight Level I and Overweight Level II, suggesting greater overlap between these adjacent weight-status categories. The model attributed substantial predictive importance to encoded gender indicators and weight, followed by vegetable consumption frequency and eating-related categorical variables. These importance values reflect predictive contributions within the model and do not imply causal relationships.

These results further show the value of integrating physical measurements with eating and lifestyle characteristics for estimating obesity levels, consistent with the dataset's intended application described by Palechor and de la Hoz Manotas (2019). The tuned XGBoost model is recommended due to its superior cross-validation performance. However, reporting Kaggle leaderboard scores is also necessary, as performance on the competition test set may differ from cross-validation results.

## References

Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. In *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 785–794). Association for Computing Machinery. https://doi.org/10.1145/2939672.2939785

Palechor, F. M., & de la Hoz Manotas, A. (2019). Dataset for estimation of obesity levels based on eating habits and physical condition in individuals from Colombia, Peru and Mexico. *Data in Brief*, 25, Article 104344. https://doi.org/10.1016/j.dib.2019.104344